# RAG Retrieval Evaluation

Unlike the previous three notebooks, this one needs **no external network access** — it evaluates this project's own ingestion pipeline and Qdrant retrieval end to end, using the `local-hash` embedding provider so it runs the same way in this sandbox as it would anywhere else. Every cell below is real, executed code against the real pipeline in `agent-service/app/rag/`.

Metric: **Hit@k** — for each evaluation question, did the correct document appear anywhere in the top-k retrieved results?

In [1]:
import sys
sys.path.insert(0, '../agent-service')

from pathlib import Path
from app.config import Settings
from app.rag.pipeline import run as run_ingestion
from app.rag import retriever

settings = Settings(
    embedding_provider='local-hash',
    embedding_dim=128,
    qdrant_local_path=str(Path('.qdrant_eval').resolve()),
    qdrant_collection='eval_collection',
)

# Isolated collection/path so this notebook doesn't depend on (or
# interfere with) whatever the dev environment's own Qdrant store
# currently holds. run_ingestion() takes settings directly rather
# than always reading the process-wide singleton.
n_chunks = run_ingestion(Path('../rag/documents'), settings=settings)
print(f'Ingested {n_chunks} chunks')

2026-09-12 01:53:09 [info     ] rag_ingest_loaded              count=9 directory=../rag/documents


2026-09-12 01:53:09 [info     ] rag_ingest_complete            chunks=27 collection=eval_collection


Ingested 27 chunks


In [2]:
eval_dataset = [
    {'question': 'kubernetes ImagePullBackOff', 'expected_document': 'kubernetes-troubleshooting'},
    {'question': 'CrashLoopBackOff pod restarting', 'expected_document': 'kubernetes-troubleshooting'},
    {'question': 'database connection refused OperationalError', 'expected_document': 'database-troubleshooting'},
    {'question': 'connection pool exhausted', 'expected_document': 'database-troubleshooting'},
    {'question': 'how to roll back a bad deployment', 'expected_document': 'rollback-procedure'},
    {'question': 'JWT token signature invalid 401', 'expected_document': 'api-authentication'},
    {'question': 'incident severity levels SEV1 SEV2', 'expected_document': 'incident-management'},
    {'question': 'action risk tiers human approval', 'expected_document': 'security-guidelines'},
]
print(f'{len(eval_dataset)} evaluation questions')

8 evaluation questions


In [3]:
def hit_at_k(question, expected_document, k=3):
    hits = retriever.search(question, settings, top_k=k)
    retrieved_docs = [h.payload['document_id'] for h in hits]
    return expected_document in retrieved_docs, retrieved_docs

results = []
for item in eval_dataset:
    hit, retrieved = hit_at_k(item['question'], item['expected_document'], k=3)
    results.append(hit)
    status = 'HIT ' if hit else 'MISS'
    print(f"{status}  {item['question']!r:55} expected={item['expected_document']!r:28} got={retrieved}")

hit_rate = sum(results) / len(results)
print(f'\nHit@3: {hit_rate:.0%} ({sum(results)}/{len(results)})')

HIT   'kubernetes ImagePullBackOff'                           expected='kubernetes-troubleshooting' got=['deployment-runbook', 'database-troubleshooting', 'kubernetes-troubleshooting']
HIT   'CrashLoopBackOff pod restarting'                       expected='kubernetes-troubleshooting' got=['gcp-production-runbook', 'gcp-production-runbook', 'kubernetes-troubleshooting']
HIT   'database connection refused OperationalError'          expected='database-troubleshooting'   got=['database-troubleshooting', 'database-troubleshooting', 'gcp-production-runbook']
HIT   'connection pool exhausted'                             expected='database-troubleshooting'   got=['database-troubleshooting', 'database-troubleshooting', 'api-authentication']
HIT   'how to roll back a bad deployment'                     expected='rollback-procedure'         got=['service-recovery', 'rollback-procedure', 'deployment-runbook']
HIT   'JWT token signature invalid 401'                       expected='api-authenticatio

## Reading these results

`local-hash` retrieval works here mainly on shared vocabulary between the question and the runbook chunk — questions phrased with the *same words* the runbook uses score well; a paraphrase with no shared vocabulary would not. This is exactly the gap a trained embedding model (`EMBEDDING_PROVIDER=huggingface`) closes: re-run this notebook with that provider on a normal-network machine and expect a materially higher hit rate on paraphrased questions, since it captures meaning rather than just shared tokens.

In [4]:
# Clean up the isolated eval collection/state used by this notebook.
import shutil
from app.rag.qdrant import _cached_client
_cached_client.cache_clear()
shutil.rmtree('.qdrant_eval', ignore_errors=True)
print('cleaned up eval state')

cleaned up eval state
